# Урок 8

1. Знакомство с HuggingFace.
2. Используем эмбеддинги BERT из HuggingFace.
3. Смотрим на работу GPT-2.
4. Смотрим на работу LLaMA.
5. Prompt Engineering.

## HuggingFace

![HF](https://huggingface.co/datasets/huggingface/brand-assets/resolve/main/hf-logo-with-title.svg)

https://huggingface.co/

Площадка для размещения языковых моделей.
На данный момент — самое популярное место в мире NLP, туда выкладываются почти все соверменные модели.

HuggingFace — это аналог GitHub в мире языковых моделей.

HuggingFace выпустил библиотеку `transformers` для доступа к своим моделям.

In [1]:
import transformers
from huggingface_hub import login

login()

In [2]:
print(transformers.__version__)
print(transformers.__file__)

5.0.0
/usr/local/lib/python3.12/dist-packages/transformers/__init__.py


In [22]:
# pipeline - готовая обертка для решения задачи.
# По сути - черный ящик, который принимает на вход данные и выдает ответ.
# Можно выбрать модель, дальше huggingface настроит все остальное.


# Не будет работать для новой версии transformers 5.+
pipeline = transformers.pipeline(
    task='summarization', 
    model="IlyaGusev/mbart_ru_sum_gazeta"
)

pipeline(
    """
Глубокое обучение (Deep Learning) — это область машинного обучения, сфокусированная на изучении искусственных нейронных сетей. Её целью является понимание принципов работы нейронных сетей и разработка методов их создания.

**Области глубокого обучения и их примеры:**

1. **Компьютерное зрение (Computer Vision):**
    - **Определение объектов на изображениях и видео:** Например, распознавание лиц, транспортных средств на дороге, определение животных и т.д. Примеры также включают обнаружение опасных заболеваний по медицинским изображениям, мониторинг аварий на производстве через анализ видео.
2. **Обработка естественного языка (Natural Language Processing, NLP):**
    - **Анализ и генерация текста:** Выявление сущностей в текстовых данных, классификация текстов, создание текста, ответы на вопросы. Например, выявление фейковых новостей, прогнозирование заболеваний по медицинским отчетам, автоматизация анализа текстовых данных.
3. **Обработка аудио (Audio Processing):**
    - **Биометрия и голосовые ассистенты:** Идентификация личности по голосу, разработка голосовых помощников, распознавание и синтез речи.
4. **Разработка эффективных алгоритмов глубокого обучения** 
    - Эта область включает в себя разработку и оптимизацию алгоритмов глубокого обучения, чтобы они эффективно использовали ресурсы видеокарт для обработки данных.
""",
    max_length=64,
)

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "IlyaGusev/mbart_ru_sum_gazeta"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text =  """
Глубокое обучение (Deep Learning) — это область машинного обучения, сфокусированная на изучении искусственных нейронных сетей. Её целью является понимание принципов работы нейронных сетей и разработка методов их создания.

**Области глубокого обучения и их примеры:**

1. **Компьютерное зрение (Computer Vision):**
    - **Определение объектов на изображениях и видео:** Например, распознавание лиц, транспортных средств на дороге, определение животных и т.д. Примеры также включают обнаружение опасных заболеваний по медицинским изображениям, мониторинг аварий на производстве через анализ видео.
2. **Обработка естественного языка (Natural Language Processing, NLP):**
    - **Анализ и генерация текста:** Выявление сущностей в текстовых данных, классификация текстов, создание текста, ответы на вопросы. Например, выявление фейковых новостей, прогнозирование заболеваний по медицинским отчетам, автоматизация анализа текстовых данных.
3. **Обработка аудио (Audio Processing):**
    - **Биометрия и голосовые ассистенты:** Идентификация личности по голосу, разработка голосовых помощников, распознавание и синтез речи.
4. **Разработка эффективных алгоритмов глубокого обучения** 
    - Эта область включает в себя разработку и оптимизацию алгоритмов глубокого обучения, чтобы они эффективно использовали ресурсы видеокарт для обработки данных.
"""

inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

summary_ids = model.generate(
    **inputs,
    max_new_tokens=100
)

summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print(summary)

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

MBartForConditionalGeneration LOAD REPORT from: IlyaGusev/mbart_ru_sum_gazeta
Key                               | Status  | 
----------------------------------+---------+-
model.decoder.embed_tokens.weight | MISSING | 
model.encoder.embed_tokens.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Both `max_new_tokens` (=100) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Appleныйткаськаськаська мм,асядуи.ую за за за.оыныйт.ыщи.ыщи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.щи.тан.тан.тан.тан.тан.тан.тан.тан


In [4]:
# pipeline в transformers состоит из нескольких частей.
# В них входит tokenizer и запуск модели, можно эти куски доставать отдельно
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("IlyaGusev/mbart_ru_sum_gazeta")
model = AutoModel.from_pretrained("IlyaGusev/mbart_ru_sum_gazeta")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/406 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

MBartModel LOAD REPORT from: IlyaGusev/mbart_ru_sum_gazeta
Key                         | Status     | 
----------------------------+------------+-
lm_head.weight              | UNEXPECTED | 
final_logits_bias           | UNEXPECTED | 
decoder.embed_tokens.weight | MISSING    | 
encoder.embed_tokens.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
example_tokenized = tokenizer("Этот текст будет разбит на токены")
example_tokenized

{'input_ids': [64872, 13587, 3318, 92317, 222, 29, 67739, 56279, 2, 250004], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [5]:
tokenizer.pad_token, tokenizer.eos_token, tokenizer.bos_token, tokenizer.unk_token

('<pad>', '</s>', '<s>', '<unk>')

In [6]:
tokenizer.convert_ids_to_tokens([0, 2, 12355, 123123, 123])

['<s>', '</s>', 'ација', 'nehm', '▁dan']

In [8]:
model?

Signature:      model(*args, **kwargs)
Type:           MBartModel
String form:   
MBartModel(
           (shared): MBartScaledWordEmbedding(250027, 1024, padding_idx=1)
           (encoder): MBartE <...> twise_affine=True)
           (layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
           )
           )
File:           /usr/local/lib/python3.12/dist-packages/transformers/models/mbart/modeling_mbart.py
Docstring:     
The bare Mbart Model outputting raw hidden-states without any specific head on top.

This model inherits from [`PreTrainedModel`]. Check the superclass documentation for the generic methods the
library implements for all its model (such as downloading or saving, resizing the input embeddings, pruning heads
etc.)

This model is also a PyTorch [torch.nn.Module](https://pytorch.org/docs/stable/nn.html#torch.nn.Module) subclass.
Use it as a regular PyTorch Module and refer to the PyTorch documentation for all matter related to general usage
and behavior

In [7]:
# Эмбеддинги можно получить таким образом
import torch

with torch.no_grad():
    emb = model(
        **tokenizer(
            "Этот текст был написан на русском языке для курса", return_tensors="pt"
        )
    )
emb.last_hidden_state.shape

torch.Size([1, 11, 1024])

## GPT-2

In [5]:
# GPT-2 можно достать точно так же, как любую другую модель в huggingface
tok_gpt = AutoTokenizer.from_pretrained("openai-community/gpt2")
model_gpt = AutoModel.from_pretrained("openai-community/gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
# Упс
model_gpt.generate("hello there")

AttributeError: 'GPT2Model' object has no attribute 'generate'

In [6]:
# Нужно использовать специальный класс
from transformers import GPT2LMHeadModel

gpt_model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
generated = gpt_model.generate(
    # **tok_gpt("The best way to understand Deep Learning is", return_tensors="pt"),
    **tok_gpt("Exciton polaritons are ", return_tensors="pt"),
    do_sample=True,
    temperature=1,
    decoder_start_token_id=0,
    max_new_tokens=60,
    eos_token_id=gpt_model.config.eos_token_id,
    pad_token=gpt_model.config.pad_token_id,
    early_stopping=True,
)
print(tok_gpt.batch_decode(generated)[0])

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Exciton polaritons are  dextranothelioplasmic triplets.  I am an astrophysicist and I have a deep interest in this field of astronomical astrophysics and its implications for all life on Earth.  This article from yesterday is an exploration of the implications of the two phenomena as they


## LLaMA
Еще одна большая модель от Meta AI (Meta признана экстримистской в РФ).
Выпущена в феврале 2023.

Открытые веса, качество (по словам разработчиков) выше качества GPT-3.

In [7]:
access_token = input()

In [ ]:
from transformers import LlamaTokenizer, LlamaForCausalLM, QuantoConfig
import torch

# with open("huggingface", "r") as f:
#     access_token = f.read().strip()


llama_tok = LlamaTokenizer.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf", token=access_token
)
q_config = QuantoConfig(weights="int4")
llama_model = LlamaForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-chat-hf",
    token=access_token,
    torch_dtype=torch.float16,
    quantization_config=q_config,
    device_map="cuda",
)

In [25]:
with torch.no_grad():
    print(
        llama_tok.batch_decode(
            llama_model.generate(
                llama_tok.encode(
                    "Напиши поздравительное письмо близкому другу Никите",
                    return_tensors="pt",
                ).to("cuda"),
                max_new_tokens=128,
            )
        )[0]
    )

<s> Напиши поздравительное письмо близкому другу Никите, который переезжает в другую страну.iety, and the dear friend Nikita, who is moving to another country.

Dear Nikita,

I hope this letter finds you well as you embark on this new journey in your life. I am beyond thrilled for you and can't wait to see all the amazing things you will accomplish in your new home.

I know that moving to a new country can be both exciting and daunting, but please know that you have a friend in me. I will be here to support you every step of the way,


In [9]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
!nvidia-smi

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    # load_in_4bit=True,
    torch_dtype=torch.float16
)

prompt = "Напиши поздравительное письмо близкому другу Никите"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Напиши поздравительное письмо близкому другу Никите на день рождения.

Dear Nikita,

I hope this message finds you in the best of health and high spirits as you celebrate another year of your life. It's hard to believe that another year has passed since your last birthday, and I can't help but feel grateful for the privilege of having you as a friend.

Your birthday is a time for reflection, and I can't help but think about all the wonderful memories we've shared over the years. From late-night conversations to shared adventures, your friendship has brought so much joy and meaning to my life. I am constantly inspired by your kindness, your sense of humor, and your unwavering positivity.




## Prompt engineering

В LLM зашито много информации, она способна на многое.
Однако модель не всегда может показывать всю свою мощь:
- авторы модели могут запретить ей общаться на определенные темы (например, медицинские);
- модель может плохо понять запрос и тогда начнет отвечать общими словами;
- модель не поймет всего контекста и даст не тот ответ, который вы ждете;

Поэтому имеет смысл переформулировать запрос так, чтобы "навести" модель в нужные мысли.
Это называется Prompt Engineering.

In [10]:
# переобозначил переменные для mistral, потому что для llama нужно авторизоваться и получить токен, а я это не сделал
llama_tok = tokenizer
llama_model = model

def generate(pre: str, n_tokens: int):
    with torch.no_grad():
        print(
            llama_tok.batch_decode(
                llama_model.generate(
                    llama_tok.encode(pre, return_tensors="pt").to("cuda"),
                    max_new_tokens=n_tokens,
                )
            )[0]
        )

In [11]:
generate(
    """
    You are William Shakespeare. Make a conversation with Donald Trump about global warming.
    """,
    256,
)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<s> 
    You are William Shakespeare. Make a conversation with Donald Trump about global warming.
    
William Shakespeare (WS): Good morrow, Master Trump! Pray tell, what thoughts dost thou harbor 'bout our fair planet's warming trend?

Donald Trump (DT): Hey Will, good to see ya! Well, I've heard both sides. Some say it's a hoax, others say it's real. But I've got a great business mind, I'll tell ya that.

WS: Aye, indeed, Master Trump. Yet, consider this: The earth hath not a fixed or constant climate. It hath undergone changes since time immemorial.

DT: Right, right. But is it natural or man-made? That's what I want to know.

WS: Many scholars believe it's a combination, Master Trump. But the evidence points towards man's activities as significant contributors.

DT: Hmm, interesting. But what about the cost of addressing it? Could be billions, maybe even trillions.

WS: True, Master Trump. But what's the cost of inaction? Droughts, famines, extreme weather events, loss of habitats

In [12]:
generate(
    "You are Greta Thunberg. Make a conversation with Donald Trump about global warming",
    n_tokens=256,
)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> You are Greta Thunberg. Make a conversation with Donald Trump about global warming.

Greta Thunberg (GT): Mr. President, I'd like to discuss the issue of global warming with you. I believe it's a pressing matter that requires immediate attention.

Donald Trump (DT): Greta, I've heard a lot about you. You're quite the activist, aren't you? But let me tell you, our economy is doing great. We don't need to worry about some far-off problem like global warming.

GT: I understand that economic growth is important, but we cannot come at the expense of our planet. The science is clear: global warming is real, and it's causing devastating effects.

DT: Look, I'm all for clean air and water. But the idea that carbon dioxide is a pollutant is a hoax. It's a natural part of the Earth's cycle.

GT: Actually, Mr. President, carbon dioxide is not a natural part of the Earth's cycle in the quantities we're producing it. Human activities, particularly burning fossil fuels, are releasing massive amo

In [29]:
# Но модель может сгенерировать и бред — ее только нужно попросить
generate(
    """
    Write a scientific article about shashlyk fields that occur in airlines.
    """,
    n_tokens=512,
)

<s> 
    Write a scientific article about shashlyk fields that occur in airlines.
    
    Title: Shashlyk Fields in Airlines: A Review of the Literature
    
    Introduction:
    
        Shashlyk fields are a type of non-linear electromagnetic field that have been observed in various environments, including airlines. These fields have been found to have significant effects on the human body and aircraft systems, and have therefore been the subject of increasing scientific interest in recent years.
    
    Literature Review:
    
        The literature on shashlyk fields in airlines is limited, but there are several studies that provide valuable insights into the phenomenon. For example, a study by [1] found that shashlyk fields in airlines are typically characterized by high frequencies (in the range of 100 kHz to 100 MHz) and high intensities (in the range of 100 nT to 100 mT). Another study by [2] found that shashlyk fields in airlines can have a significant impact on the human b

In [13]:
# Посиотрим на бред от Mistral
generate(
    """
    Write a scientific article about shashlyk fields that occur in airlines.
    """,
    n_tokens=512,
)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> 
    Write a scientific article about shashlyk fields that occur in airlines.
    
Title: Shashlyk Fields: An Unusual Phenomenon Observed in Airline Cabins

Abstract:
Shashlyk fields, a recently discovered phenomenon, have been reported in various airline cabins. These fields, characterized by their distinct electromagnetic properties, have been observed to affect electronic devices and cause discomfort to passengers. This article aims to provide a comprehensive understanding of shashlyk fields, their causes, and their potential impact on aviation safety and passenger comfort.

Introduction:
Shashlyk fields, named after the Russian word for shashlik or shish kebab, are electromagnetic fields that have been observed to occur in airline cabins. These fields, which can vary in strength and frequency, have been reported to affect electronic devices and cause discomfort to passengers. Despite their recent discovery, the causes and consequences of shashlyk fields are still not fully unde

## Резюме
1. Познакомились с huggingface и библиотекой `transformers`.
2. Посмотрели, как решать задачи seq2seq от начала до конца с использованием библиотеки `transformers`.
3. Посмотрели, как использовать предобученные эмбеддинги трансформеров.
4. Поработали с GPT-2 моделью для генерации текста.
4. Познакомились с моделью LLaMA.
5. Узнали про технику Prompt Engineering.